# NumPy ML Coding Interview Review

**Goal**: Master NumPy patterns essential for ML/MLE coding interviews

**Focus**: Vectorization, broadcasting, shape reasoning, and core ML algorithms

**Audience**: ML/MLE interview candidates preparing for coding rounds

---


## 1. NumPy Basics for ML

### Intuition
NumPy arrays are the foundation of ML in Python. Understanding shapes, dimensions, and axis semantics is critical for debugging and implementing algorithms correctly.

### Why This Matters in ML Interviews
- **Shape discipline**: ML pipelines require precise shape management (n_samples, n_features)
- **Axis semantics**: Operations like mean, sum, and concatenation depend on correct axis specification
- **Memory efficiency**: Understanding reshape vs copy operations affects performance


In [1]:
import numpy as np

# Create a toy dataset: (n_samples, n_features)
# 5 samples, 3 features
X = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10.0, 11.0, 12.0],
    [13.0, 14.0, 15.0]
])

print(f"Shape: {X.shape}")  # (5, 3) - 5 samples, 3 features
print(f"Number of dimensions: {X.ndim}")  # 2
print(f"Data type: {X.dtype}")  # float64
print(f"Total elements: {X.size}")  # 15

# Reshape: changes view (if possible) or creates copy
X_reshaped = X.reshape(3, 5)  # (3, 5)
print(f"\nReshaped to (3, 5):\n{X_reshaped}")

# Ravel: returns flattened view (if possible)
X_flat = X.ravel()  # shape: (15,)
print(f"\nRaveled (view): {X_flat}")

# Flatten: always returns a copy
X_flat_copy = X.flatten()  # shape: (15,)
print(f"Flattened (copy): {X_flat_copy}")

# Axis semantics: critical for ML operations
print(f"\nMean along axis=0 (across samples, per feature): {X.mean(axis=0)}")  # shape: (3,)
print(f"Mean along axis=1 (across features, per sample): {X.mean(axis=1)}")  # shape: (5,)
print(f"Mean of entire array: {X.mean()}")  # scalar


Shape: (5, 3)
Number of dimensions: 2
Data type: float64
Total elements: 15

Reshaped to (3, 5):
[[ 1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10.]
 [11. 12. 13. 14. 15.]]

Raveled (view): [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Flattened (copy): [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]

Mean along axis=0 (across samples, per feature): [7. 8. 9.]
Mean along axis=1 (across features, per sample): [ 2.  5.  8. 11. 14.]
Mean of entire array: 8.0


### Broadcasting Deep Dive: Why (3,) + (3, 1) = (3, 3)?

**Key Rule**: NumPy aligns dimensions from the **right** and broadcasts missing dimensions.

When you do `(3,) + (3, 1)`:
1. NumPy treats `(3,)` as `(1, 3)` for broadcasting (adds a leading dimension)
2. Now we have `(1, 3)` + `(3, 1)`
3. NumPy "stretches" both arrays to make them compatible:
   - `(1, 3)` → `(3, 3)` by repeating the row 3 times
   - `(3, 1)` → `(3, 3)` by repeating the column 3 times
4. Result: `(3, 3)` - every combination of row + column

**Visual Example**:
```
row = [1, 2, 3]        shape: (3,) → treated as [[1, 2, 3]]  shape: (1, 3)
col = [[1], [2], [3]]  shape: (3, 1)

After broadcasting:
row becomes: [[1, 2, 3],    col becomes: [[1, 1, 1],
              [1, 2, 3],                  [2, 2, 2],
              [1, 2, 3]]                  [3, 3, 3]]

Result: [[2, 3, 4],
         [3, 4, 5],
         [4, 5, 6]]
```

**To get (3, 1) result**, you need compatible shapes:
- `(3, 1) + (3, 1)` = `(3, 1)` ✅
- `(3, 1) + (1, 1)` = `(3, 1)` ✅ (scalar broadcast)
- `(3,) + (3, 1)` = `(3, 3)` ❌ (creates outer product)


In [11]:
# Demonstration: Why (3,) + (3, 1) = (3, 3) and not (3, 1)

row = np.array([1, 2, 3])  # shape: (3,)
col = np.array([[1], [2], [3]])  # shape: (3, 1)

print("Original shapes:")
print(f"row: {row.shape} = {row}")
print(f"col: {col.shape} = \n{col}")

print("\n" + "="*50)
print("What happens during broadcasting:")
print("="*50)

# NumPy aligns from the right and adds missing dimensions
# (3,) is treated as (1, 3) for broadcasting
print(f"\n1. row shape (3,) is treated as (1, 3) for broadcasting")
print(f"2. We have: (1, 3) + (3, 1)")
print(f"3. NumPy stretches both:")
print(f"   - (1, 3) → (3, 3) by repeating the row")
print(f"   - (3, 1) → (3, 3) by repeating the column")

result = row + col
print(f"\nResult shape: {result.shape}")
print(f"Result:\n{result}")

print("\n" + "="*50)
print("To get (3, 1) result, use compatible shapes:")
print("="*50)

# Option 1: Both (3, 1)
col1 = np.array([[1], [2], [3]])  # (3, 1)
col2 = np.array([[10], [20], [30]])  # (3, 1)
result_3x1 = col1 + col2
print(f"\n(3, 1) + (3, 1) = {result_3x1.shape}")
print(f"Result:\n{result_3x1}")

# Option 2: (3, 1) + scalar (which broadcasts to (1, 1))
scalar = 5
result_scalar = col1 + scalar
print(f"\n(3, 1) + scalar = {result_scalar.shape}")
print(f"Result:\n{result_scalar}")

# Option 3: Reshape row to (3, 1) first
row_reshaped = row.reshape(3, 1)  # (3, 1)
result_reshaped = row_reshaped + col1
print(f"\n(3, 1) + (3, 1) (after reshape) = {result_reshaped.shape}")
print(f"Result:\n{result_reshaped}")


Original shapes:
row: (3,) = [1 2 3]
col: (3, 1) = 
[[1]
 [2]
 [3]]

What happens during broadcasting:

1. row shape (3,) is treated as (1, 3) for broadcasting
2. We have: (1, 3) + (3, 1)
3. NumPy stretches both:
   - (1, 3) → (3, 3) by repeating the row
   - (3, 1) → (3, 3) by repeating the column

Result shape: (3, 3)
Result:
[[2 3 4]
 [3 4 5]
 [4 5 6]]

To get (3, 1) result, use compatible shapes:

(3, 1) + (3, 1) = (3, 1)
Result:
[[11]
 [22]
 [33]]

(3, 1) + scalar = (3, 1)
Result:
[[6]
 [7]
 [8]]

(3, 1) + (3, 1) (after reshape) = (3, 1)
Result:
[[2]
 [4]
 [6]]


## 2. Vectorization & Broadcasting

### Intuition
Vectorization replaces Python loops with NumPy operations, making code 10-100x faster. Broadcasting allows operations between arrays of different shapes without explicit loops.

### Why This Matters in ML Interviews
- **Performance**: Interviewers expect vectorized solutions, not Python loops
- **Broadcasting**: Essential for operations like feature scaling, normalization, and distance computations
- **Code quality**: Vectorized code is cleaner and more "NumPy-idiomatic"


In [2]:
# Elementwise operations (automatic broadcasting)
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(f"Elementwise addition: {a + b}")  # [5, 7, 9]
print(f"Elementwise multiplication: {a * b}")  # [4, 10, 18]

# Broadcasting: smaller array is "stretched" to match larger array
# Shape (3,) broadcasts with (3, 1) or (1, 3)
row = np.array([1, 2, 3])  # shape: (3,)
col = np.array([[1], [2], [3]])  # shape: (3, 1)
print(f"\nBroadcasting (3,) + (3, 1):\n{row + col}")  # shape: (3, 3)

# Feature scaling example: Standardization
# X shape: (n_samples, n_features)
X = np.random.randn(100, 5)  # 100 samples, 5 features

# SLOW: Python loop (don't do this in interviews!)
def scale_loop(X):
    X_scaled = np.zeros_like(X)
    for i in range(X.shape[1]):
        mean = X[:, i].mean()
        std = X[:, i].std()
        X_scaled[:, i] = (X[:, i] - mean) / std
    return X_scaled

# FAST: Vectorized (this is what interviewers want!)
def scale_vectorized(X):
    mean = X.mean(axis=0)  # shape: (n_features,)
    std = X.std(axis=0)    # shape: (n_features,)
    return (X - mean) / std  # Broadcasting: (n_samples, n_features) - (n_features,)

X_scaled_loop = scale_loop(X)
X_scaled_vec = scale_vectorized(X)

print(f"Results match: {np.allclose(X_scaled_loop, X_scaled_vec)}")
print(f"Vectorized version is ~100x faster for large arrays!")


Elementwise addition: [5 7 9]
Elementwise multiplication: [ 4 10 18]

Broadcasting (3,) + (3, 1):
[[2 3 4]
 [3 4 5]
 [4 5 6]]
Results match: True
Vectorized version is ~100x faster for large arrays!


## 3. Dot Product & Matrix Multiplication

### Intuition
The dot product (matrix multiplication) is the computational core of linear models. Understanding `@` operator and `np.dot` is essential for implementing ML algorithms.

### Why This Matters in ML Interviews
- **Linear models**: All linear models (linear regression, logistic regression, neural networks) use matrix multiplication
- **Efficiency**: NumPy's optimized BLAS backend makes matrix multiplication extremely fast
- **Shape reasoning**: Getting shapes right is critical (e.g., (n, d) @ (d, 1) = (n, 1))


In [3]:
# Vector-vector dot product (scalar result)
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
dot_scalar = np.dot(v1, v2)  # 1*4 + 2*5 + 3*6 = 32
print(f"Vector dot product: {dot_scalar}")

# Matrix-vector multiplication
# X shape: (n_samples, n_features), w shape: (n_features,)
# Result: (n_samples,)
X = np.array([[1, 2], [3, 4], [5, 6]])  # (3, 2)
w = np.array([0.5, 1.0])  # (2,)
y = X @ w  # (3, 2) @ (2,) = (3,)
print(f"\nMatrix-vector multiplication:\nX @ w = {y}")  # [2.5, 5.5, 8.5]

# Matrix-matrix multiplication
# X shape: (n_samples, n_features), W shape: (n_features, n_outputs)
# Result: (n_samples, n_outputs)
W = np.array([[0.5, 1.0], [1.5, 2.0]])  # (2, 2)
Y = X @ W  # (3, 2) @ (2, 2) = (3, 2)
print(f"\nMatrix-matrix multiplication:\nX @ W =\n{Y}")

# Linear regression prediction: y = X @ w + b
# X: (n_samples, n_features)
# w: (n_features,)
# b: scalar
# y: (n_samples,)
n_samples, n_features = 100, 5
X = np.random.randn(n_samples, n_features)
w = np.random.randn(n_features)
b = 0.5
y_pred = X @ w + b  # Broadcasting: scalar b added to (n_samples,)
print(f"\nLinear regression prediction shape: {y_pred.shape}")

# Logistic regression logits: logits = X @ w + b
# Then apply sigmoid: p = 1 / (1 + exp(-logits))
logits = X @ w + b
p = 1 / (1 + np.exp(-logits))  # Sigmoid applied elementwise
print(f"Logistic regression probabilities shape: {p.shape}")
print(f"Probabilities range: [{p.min():.3f}, {p.max():.3f}]")


Vector dot product: 32

Matrix-vector multiplication:
X @ w = [2.5 5.5 8.5]

Matrix-matrix multiplication:
X @ W =
[[ 3.5  5. ]
 [ 7.5 11. ]
 [11.5 17. ]]

Linear regression prediction shape: (100,)
Logistic regression probabilities shape: (100,)
Probabilities range: [0.065, 0.987]


## 4. Norms & Distances (VERY IMPORTANT)

### Intuition
Norms measure the "size" of vectors. L2 norm (Euclidean) and L1 norm (Manhattan) are fundamental for distance computations, regularization, and clustering algorithms.

### Why This Matters in ML Interviews
- **KMeans**: Requires computing distances between points and centroids
- **KNN**: Uses distance metrics to find nearest neighbors
- **Regularization**: L1 (Lasso) and L2 (Ridge) regularization use norms
- **Optimization**: Gradient descent uses norms to measure convergence


In [4]:
# L2 norm (Euclidean norm) of a vector
x = np.array([3, 4])
l2_norm = np.linalg.norm(x)  # sqrt(3^2 + 4^2) = 5.0
print(f"L2 norm of {x}: {l2_norm}")

# L1 norm (Manhattan norm)
l1_norm = np.linalg.norm(x, ord=1)  # |3| + |4| = 7.0
print(f"L1 norm of {x}: {l1_norm}")

# Distance between two vectors (L2 distance)
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
distance = np.linalg.norm(x - y)  # sqrt((1-4)^2 + (2-5)^2 + (3-6)^2) = sqrt(27)
print(f"\nL2 distance between x and y: {distance:.3f}")

# Squared L2 distance (often used to avoid sqrt computation)
# Used in KMeans, KNN when only comparing distances (not absolute values)
squared_l2 = np.sum((x - y) ** 2)  # 27 (no sqrt)
print(f"Squared L2 distance: {squared_l2}")

# Why squared L2? argmin works the same, but faster (no sqrt)
distances = np.array([5.0, 3.0, 7.0])
squared_distances = np.array([25.0, 9.0, 49.0])
print(f"\nOriginal distances: {distances}")
print(f"Argmin (closest): {np.argmin(distances)}")
print(f"Squared distances: {squared_distances}")
print(f"Argmin (same result): {np.argmin(squared_distances)}")

# Norm along specific axis
X = np.array([[1, 2], [3, 4], [5, 6]])  # (3, 2)
norms_per_sample = np.linalg.norm(X, axis=1)  # shape: (3,)
print(f"\nL2 norm per sample:\n{norms_per_sample}")


L2 norm of [3 4]: 5.0
L1 norm of [3 4]: 7.0

L2 distance between x and y: 5.196
Squared L2 distance: 27

Original distances: [5. 3. 7.]
Argmin (closest): 1
Squared distances: [25.  9. 49.]
Argmin (same result): 1

L2 norm per sample:
[2.23606798 5.         7.81024968]


## 5. Cosine Similarity (Embeddings / Retrieval)

### Intuition
Cosine similarity measures the angle between two vectors, independent of magnitude. It's the go-to metric for embeddings, recommendation systems, and retrieval tasks.

### Why This Matters in ML Interviews
- **Embeddings**: Word embeddings, sentence embeddings, image embeddings all use cosine similarity
- **Recommendation systems**: "Users who liked X also liked Y" uses cosine similarity
- **Information retrieval**: Search engines rank by cosine similarity between query and documents
- **Interview frequency**: Very common in ML/NLP interviews


In [5]:
# Cosine similarity formula: cos(θ) = (a · b) / (||a|| * ||b||)
# Range: [-1, 1], where 1 = identical direction, 0 = orthogonal, -1 = opposite

def cosine_similarity(a, b):
    """
    Compute cosine similarity between two vectors.
    
    Args:
        a: array of shape (n_features,)
        b: array of shape (n_features,)
    
    Returns:
        scalar: cosine similarity in [-1, 1]
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# Example: word embeddings
word1 = np.array([1, 2, 3])  # embedding for "king"
word2 = np.array([2, 4, 6])  # embedding for "queen" (similar direction)
word3 = np.array([-1, -2, -3])  # opposite direction

cos_sim_12 = cosine_similarity(word1, word2)
cos_sim_13 = cosine_similarity(word1, word3)

print(f"Cosine similarity (word1, word2): {cos_sim_12:.3f}")  # ~1.0 (same direction)
print(f"Cosine similarity (word1, word3): {cos_sim_13:.3f}")  # -1.0 (opposite)

# Batch cosine similarity: compute similarity between query and all documents
# query shape: (n_features,)
# documents shape: (n_docs, n_features)
# result shape: (n_docs,)
query = np.array([1, 0, 1])
documents = np.array([
    [1, 0, 1],    # identical to query
    [0, 1, 0],    # orthogonal
    [-1, 0, -1],  # opposite
])

# Vectorized batch computation
dot_products = documents @ query  # (n_docs, n_features) @ (n_features,) = (n_docs,)
norms_docs = np.linalg.norm(documents, axis=1)  # (n_docs,)
norm_query = np.linalg.norm(query)  # scalar
cosine_similarities = dot_products / (norms_docs * norm_query)  # (n_docs,)

print(f"\nQuery: {query}")
print(f"Cosine similarities with documents:\n{cosine_similarities}")

# Compare cosine similarity vs L2 distance
# Cosine similarity: measures angle (direction)
# L2 distance: measures Euclidean distance (magnitude + direction)
a = np.array([1, 1])
b = np.array([2, 2])  # same direction, different magnitude
c = np.array([1, 0])  # different direction, similar magnitude

print(f"\nCosine similarity (a, b): {cosine_similarity(a, b):.3f}")  # 1.0 (same direction)
print(f"L2 distance (a, b): {np.linalg.norm(a - b):.3f}")  # 1.414 (different magnitude)

print(f"Cosine similarity (a, c): {cosine_similarity(a, c):.3f}")  # 0.707 (45° angle)
print(f"L2 distance (a, c): {np.linalg.norm(a - c):.3f}")  # 1.0 (similar magnitude)


Cosine similarity (word1, word2): 1.000
Cosine similarity (word1, word3): -1.000

Query: [1 0 1]
Cosine similarities with documents:
[ 1.  0. -1.]

Cosine similarity (a, b): 1.000
L2 distance (a, b): 1.414
Cosine similarity (a, c): 0.707
L2 distance (a, c): 1.000


In [13]:
# Goal: Compute pairwise distances between X and centroids
# X shape: (n_samples, n_features)
# centroids shape: (k, n_features)
# Output: distances shape: (n_samples, k)

n_samples, n_features = 100, 3
k = 5

X = np.random.randn(n_samples, n_features)
centroids = np.random.randn(k, n_features)

# Method 1: Using None / np.newaxis for broadcasting
# X[:, None, :] expands to (n_samples, 1, n_features)
# centroids[None, :, :] expands to (1, k, n_features)
# Broadcasting: (n_samples, 1, n_features) - (1, k, n_features) = (n_samples, k, n_features)
# Then compute norm along axis=2 to get (n_samples, k)

distances = np.linalg.norm(
    X[:, None, :] - centroids[None, :, :],
    axis=2
)
print(f"X shape: {X.shape}")
print(f"Centroids shape: {centroids.shape}")
print(f"Distances shape: {distances.shape}")  # (100, 5)

# Visualize the broadcasting
print(f"\nX[:, None, :] shape: {X[:, None, :].shape}")  # (100, 1, 3)
print(f"centroids[None, :, :] shape: {centroids[None, :, :].shape}")  # (1, 5, 3)
print(f"After subtraction: {(X[:, None, :] - centroids[None, :, :]).shape}")  # (100, 5, 3)

# Method 2: Alternative using reshape (less common but equivalent)
X_expanded = X.reshape(n_samples, 1, n_features)  # (n_samples, 1, n_features)
centroids_expanded = centroids.reshape(1, k, n_features)  # (1, k, n_features)
distances_alt = np.linalg.norm(X_expanded - centroids_expanded, axis=2)
print(f"\nAlternative method matches: {np.allclose(distances, distances_alt)}")

# Example: Find closest centroid for each point
closest_centroid = np.argmin(distances, axis=1)  # shape: (n_samples,)
print(f"\nClosest centroid indices (first 10): {closest_centroid[:10]}")

# Example: Squared L2 distance (faster, same argmin result)
squared_distances = np.sum(
    (X[:, None, :] - centroids[None, :, :]) ** 2,
    axis=2
)
closest_centroid_squared = np.argmin(squared_distances, axis=1)
print(f"Using squared distances gives same result: {np.array_equal(closest_centroid, closest_centroid_squared)}")


X shape: (100, 3)
Centroids shape: (5, 3)
Distances shape: (100, 5)

X[:, None, :] shape: (100, 1, 3)
centroids[None, :, :] shape: (1, 5, 3)
After subtraction: (100, 5, 3)

Alternative method matches: True

Closest centroid indices (first 10): [2 2 3 2 4 2 1 4 3 3]
Using squared distances gives same result: True


## 7. KMeans Core Computation (No Full Class)

### Intuition
KMeans alternates between: (1) assigning points to nearest centroids, (2) updating centroids as mean of assigned points. The NumPy implementation tests broadcasting, argmin, and mean operations.

### Why This Matters in ML Interviews
- **Algorithm understanding**: Shows you understand KMeans beyond sklearn
- **NumPy mastery**: Tests broadcasting, indexing, and aggregation skills
- **Common interview question**: "Implement one iteration of KMeans"


In [7]:
# KMeans: One iteration step-by-step
# Input: X shape (n_samples, n_features), k centroids shape (k, n_features)
# Output: Updated centroids, cluster assignments

np.random.seed(42)
n_samples, n_features = 200, 2
k = 3

# Generate toy data
X = np.random.randn(n_samples, n_features)

# Initialize centroids randomly
centroids = np.random.randn(k, n_features)

print(f"Initial centroids shape: {centroids.shape}")

# Step 1: Compute distances from all points to all centroids
# distances shape: (n_samples, k)
distances = np.linalg.norm(
    X[:, None, :] - centroids[None, :, :],
    axis=2
)

# Step 2: Assign each point to nearest centroid
# assignments shape: (n_samples,)
assignments = np.argmin(distances, axis=1)
print(f"\nCluster assignments (first 10): {assignments[:10]}")

# Step 3: Update centroids as mean of assigned points
# For each cluster k, compute mean of X[assignments == k]
new_centroids = np.zeros_like(centroids)
for i in range(k):
    mask = assignments == i  # boolean array, shape: (n_samples,)
    if mask.sum() > 0:  # avoid division by zero
        new_centroids[i] = X[mask].mean(axis=0)  # mean along samples
    else:
        new_centroids[i] = centroids[i]  # keep old centroid if no points assigned

print(f"\nUpdated centroids:\n{new_centroids}")

# Vectorized centroid update (more advanced, but cleaner)
# This uses advanced indexing - less common in interviews but shows mastery
new_centroids_vec = np.array([
    X[assignments == i].mean(axis=0) if (assignments == i).sum() > 0 
    else centroids[i]
    for i in range(k)
])
print(f"Vectorized update matches: {np.allclose(new_centroids, new_centroids_vec)}")

# What interviewers look for:
# 1. Correct distance computation using broadcasting
# 2. Correct use of argmin for assignment
# 3. Correct mean computation for centroid update
# 4. Handling edge case (empty clusters)


Initial centroids shape: (3, 2)

Cluster assignments (first 10): [1 2 1 1 2 1 1 0 2 0]

Updated centroids:
[[-1.34071874 -0.58974686]
 [ 0.55972352 -0.32603232]
 [-0.35115241  0.99536413]]
Vectorized update matches: True


## 8. KNN Distance & Voting

### Intuition
KNN finds the k nearest neighbors by computing distances and using argsort to select top-k. Then it votes (classification) or averages (regression) among neighbors.

### Why This Matters in ML Interviews
- **Distance computation**: Tests broadcasting skills
- **Indexing**: argsort, fancy indexing, and slicing are common interview topics
- **Algorithm implementation**: Shows understanding beyond sklearn API


In [8]:
# KNN: Predict for one query point
# X_train shape: (n_train, n_features)
# y_train shape: (n_train,) for classification
# query shape: (n_features,)
# k: number of neighbors

np.random.seed(42)
n_train, n_features = 100, 3
k = 5

X_train = np.random.randn(n_train, n_features)
y_train = np.random.randint(0, 3, size=n_train)  # 3 classes: 0, 1, 2
query = np.random.randn(n_features)

print(f"Training data shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Query shape: {query.shape}")

# Step 1: Compute distances from query to all training points
# query shape: (n_features,)
# X_train shape: (n_train, n_features)
# distances shape: (n_train,)

distances = np.linalg.norm(X_train - query, axis=1)  # Broadcasting: (n_train, n_features) - (n_features,)
print(f"\nDistances shape: {distances.shape}")
print(f"First 10 distances: {distances[:10]}")

# Step 2: Find k nearest neighbors using argsort
# argsort returns indices that would sort the array
k_nearest_indices = np.argsort(distances)[:k]  # shape: (k,)
print(f"\nK nearest neighbor indices: {k_nearest_indices}")
print(f"K nearest distances: {distances[k_nearest_indices]}")

# Step 3: Get labels of k nearest neighbors
k_nearest_labels = y_train[k_nearest_indices]  # shape: (k,)
print(f"K nearest labels: {k_nearest_labels}")

# Step 4: Majority vote (classification)
# Count occurrences of each class
unique_labels, counts = np.unique(k_nearest_labels, return_counts=True)
predicted_label = unique_labels[np.argmax(counts)]
print(f"\nPredicted label: {predicted_label}")

# Alternative: Using bincount (more efficient for integer labels)
if y_train.dtype == int:
    label_counts = np.bincount(k_nearest_labels)
    predicted_label_bincount = np.argmax(label_counts)
    print(f"Using bincount: {predicted_label_bincount}")

# For regression: average the target values
y_train_regression = np.random.randn(n_train)  # continuous targets
k_nearest_targets = y_train_regression[k_nearest_indices]
predicted_value = k_nearest_targets.mean()
print(f"\nRegression prediction (mean of k neighbors): {predicted_value:.3f}")

# What interviewers test:
# 1. Distance computation (broadcasting)
# 2. argsort for finding top-k
# 3. Fancy indexing to get neighbor labels
# 4. Voting or averaging logic


Training data shape: (100, 3)
Labels shape: (100,)
Query shape: (3,)

Distances shape: (100,)
First 10 distances: [0.75845015 0.70320859 1.25840087 0.62525222 2.52257621 1.68959767
 2.62493572 1.78931744 1.82221329 0.70771227]

K nearest neighbor indices: [25  3 32  1  9]
K nearest distances: [0.37506732 0.62525222 0.70224245 0.70320859 0.70771227]
K nearest labels: [1 0 2 1 2]

Predicted label: 1
Using bincount: 1

Regression prediction (mean of k neighbors): 0.406


## 9. Logistic Regression Loss with NumPy

### Intuition
Binary cross-entropy loss measures how well predicted probabilities match true labels. The sigmoid function maps logits to probabilities, and the loss is computed elementwise then averaged.

### Why This Matters in ML Interviews
- **Loss functions**: Understanding loss computation is fundamental
- **Sigmoid**: Core activation function for binary classification
- **Vectorization**: Must compute loss over entire dataset efficiently
- **Gradient computation**: Often asked to compute gradients next


In [9]:
# Sigmoid function: σ(z) = 1 / (1 + exp(-z))
# Maps logits to probabilities in [0, 1]

def sigmoid(z):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-z))

# Test sigmoid
logits = np.array([-2, 0, 2])
probs = sigmoid(logits)
print(f"Logits: {logits}")
print(f"Probabilities: {probs}")  # [0.119, 0.5, 0.881]

# Binary cross-entropy loss
# L = -[y * log(p) + (1-y) * log(1-p)]
# where p = sigmoid(X @ w + b), y is true label

def binary_cross_entropy_loss(X, y, w, b):
    """
    Compute binary cross-entropy loss.
    
    Args:
        X: shape (n_samples, n_features)
        y: shape (n_samples,), binary labels {0, 1}
        w: shape (n_features,)
        b: scalar
    
    Returns:
        scalar: average loss
    """
    # Compute logits and probabilities
    logits = X @ w + b  # shape: (n_samples,)
    probs = sigmoid(logits)  # shape: (n_samples,)
    
    # Avoid log(0) by clipping probabilities
    eps = 1e-15
    probs = np.clip(probs, eps, 1 - eps)
    
    # Compute loss elementwise
    loss_per_sample = -(y * np.log(probs) + (1 - y) * np.log(1 - probs))
    
    # Average over samples
    return loss_per_sample.mean()

# Example
np.random.seed(42)
n_samples, n_features = 100, 5
X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 2, size=n_samples)  # binary labels
w = np.random.randn(n_features)
b = 0.5

loss = binary_cross_entropy_loss(X, y, w, b)
print(f"\nBinary cross-entropy loss: {loss:.4f}")

# Verify with sklearn (for sanity check, but don't use in interviews)
from sklearn.metrics import log_loss
logits_sklearn = X @ w + b
probs_sklearn = sigmoid(logits_sklearn)
loss_sklearn = log_loss(y, probs_sklearn)
print(f"Sklearn verification: {loss_sklearn:.4f}")
print(f"Match: {np.allclose(loss, loss_sklearn)}")

# What interviewers look for:
# 1. Correct sigmoid implementation
# 2. Correct loss formula
# 3. Numerical stability (clipping to avoid log(0))
# 4. Vectorized computation (no loops)


Logits: [-2  0  2]
Probabilities: [0.11920292 0.5        0.88079708]

Binary cross-entropy loss: 1.5223
Sklearn verification: 1.5223
Match: True


## 10. Metrics & Evaluation (Lightweight)

### Intuition
Implementing metrics from scratch demonstrates understanding of evaluation fundamentals and NumPy operations like comparison, mean, and sum.

### Why This Matters in ML Interviews
- **Fundamentals**: Shows you understand metrics beyond sklearn API
- **NumPy operations**: Tests boolean indexing, mean, sum
- **Common question**: "Implement accuracy/MSE without sklearn"


In [10]:
# Accuracy: fraction of correct predictions
def accuracy(y_true, y_pred):
    """Compute accuracy for classification."""
    return (y_true == y_pred).mean()

# Example
y_true = np.array([0, 1, 1, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1])
acc = accuracy(y_true, y_pred)
print(f"Accuracy: {acc:.2f}")  # 0.8 (4 out of 5 correct)

# Mean Squared Error (MSE)
def mse(y_true, y_pred):
    """Compute mean squared error for regression."""
    return np.mean((y_true - y_pred) ** 2)

# Example
y_true_reg = np.array([1.0, 2.0, 3.0, 4.0])
y_pred_reg = np.array([1.1, 1.9, 3.2, 3.8])
mse_value = mse(y_true_reg, y_pred_reg)
print(f"\nMSE: {mse_value:.4f}")

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    """Compute root mean squared error."""
    return np.sqrt(mse(y_true, y_pred))

rmse_value = rmse(y_true_reg, y_pred_reg)
print(f"RMSE: {rmse_value:.4f}")

# Cosine similarity as metric (for embeddings)
def cosine_similarity_metric(query, documents):
    """
    Compute cosine similarity between query and all documents.
    
    Args:
        query: shape (n_features,)
        documents: shape (n_docs, n_features)
    
    Returns:
        similarities: shape (n_docs,)
    """
    dot_products = documents @ query  # (n_docs,)
    norms_docs = np.linalg.norm(documents, axis=1)  # (n_docs,)
    norm_query = np.linalg.norm(query)  # scalar
    return dot_products / (norms_docs * norm_query)

# Example: find most similar document
query = np.array([1, 0, 1])
documents = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [-1, 0, -1],
    [2, 0, 2],
])

similarities = cosine_similarity_metric(query, documents)
most_similar_idx = np.argmax(similarities)
print(f"\nCosine similarities: {similarities}")
print(f"Most similar document index: {most_similar_idx}")

# Confusion matrix (for multi-class classification)
def confusion_matrix(y_true, y_pred, n_classes):
    """
    Compute confusion matrix.
    
    Returns:
        matrix: shape (n_classes, n_classes)
        matrix[i, j] = count of samples with true label i predicted as j
    """
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    for i in range(len(y_true)):
        matrix[y_true[i], y_pred[i]] += 1
    return matrix

y_true_multi = np.array([0, 1, 2, 0, 1, 2])
y_pred_multi = np.array([0, 1, 1, 0, 2, 2])
cm = confusion_matrix(y_true_multi, y_pred_multi, n_classes=3)
print(f"\nConfusion matrix:\n{cm}")

# What interviewers look for:
# 1. Correct formula implementation
# 2. Vectorized operations (no loops where possible)
# 3. Understanding of shape semantics
# 4. Numerical stability considerations


Accuracy: 0.80

MSE: 0.0250
RMSE: 0.1581

Cosine similarities: [ 1.  0. -1.  1.]
Most similar document index: 0

Confusion matrix:
[[2 0 0]
 [0 1 1]
 [0 1 1]]


## 11. Interview Pattern Cheat Sheet

### Quick Reference Table

| Task | NumPy Pattern | Shape Notes |
|------|---------------|-------------|
| **Linear model prediction** | `y = X @ w + b` | `(n, d) @ (d,) + scalar = (n,)` |
| **Cosine similarity** | `dot / (norm_a * norm_b)` | `(d,) · (d,) → scalar` |
| **KMeans distance** | `norm(X[:, None, :] - C[None, :, :], axis=2)` | `(n, d) → (n, k)` |
| **Nearest neighbor** | `argsort(distances)[:k]` | `(n,) → (k,)` |
| **Squared L2 distance** | `np.sum((x - y) ** 2)` | `(d,) - (d,) → scalar` |
| **Feature scaling** | `(X - mean) / std` | Broadcasting: `(n, d) - (d,)` |
| **Sigmoid** | `1 / (1 + exp(-z))` | Elementwise: `(n,) → (n,)` |
| **Binary cross-entropy** | `-mean(y*log(p) + (1-y)*log(1-p))` | `(n,) → scalar` |
| **Accuracy** | `(y_true == y_pred).mean()` | `(n,) → scalar` |
| **MSE** | `mean((y_true - y_pred) ** 2)` | `(n,) → scalar` |

### Key Broadcasting Patterns

1. **Row-wise operations**: `X - X.mean(axis=0)` → subtract mean per feature
2. **Column-wise operations**: `X - X.mean(axis=1, keepdims=True)` → subtract mean per sample
3. **Pairwise distances**: `X[:, None, :] - Y[None, :, :]` → all pairs
4. **Outer product**: `a[:, None] * b[None, :]` → `(n,) × (m,) → (n, m)`

### Common Shape Patterns

- **Dataset**: `(n_samples, n_features)`
- **Weights**: `(n_features,)` or `(n_features, n_outputs)`
- **Predictions**: `(n_samples,)` or `(n_samples, n_outputs)`
- **Distances**: `(n_samples, k)` for k neighbors/centroids
- **Loss**: scalar (average over samples)

### Axis Semantics Reminder

- `axis=0`: along rows (across samples) → reduces first dimension
- `axis=1`: along columns (across features) → reduces second dimension
- `keepdims=True`: preserves dimension for broadcasting

### Performance Tips

1. **Use squared L2** when only comparing distances (no sqrt needed)
2. **Avoid loops** - vectorize everything possible
3. **Use `np.clip`** for numerical stability (e.g., in log loss)
4. **Prefer `@` over `np.dot`** for matrix multiplication (cleaner syntax)

---

**Good luck with your interviews! 🚀**
